# **Training and Downloading the model**
In this notebook we will showcase how the model is made.
Firstly, we will need to download the model then download the dataset. Create Histograms. Add OOD label to the dataset.
Save the model.

**NOTE** K-NN is a LAZY learner. So technically we are not training we are making vocab

## **Environment Preperation**

### **Imports**

In [21]:
import torch
import numpy as np
import torchvision
import matplotlib.pyplot as plt
import kagglehub
from transformers import AutoImageProcessor, AutoModel
import os
import torchvision.transforms as transforms

### **Cuda**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## **Model Import**

### **1.1 Hugging Face DinoV3**
If you have access to DinoV3 from Hugging Face. You can download a model size that your computer can withstand.

For access and model information visit this [link](https://huggingface.co/facebook/dinov3-vitb16-pretrain-lvd1689m)


In [ ]:
# Change the model name to a model you have opted for
model_name = "facebook/dinov3-vit7b16-pretrain-lvd1689m"

In [ ]:
if 'dtype' not in locals():
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32

### **1.2 Log In**

In [ ]:
from huggingface_hub import login
login()

### **1.3 Model Download**

In [ ]:
try:
    #Load Model with Memory Optimization and Eager Attention
    # We strictly need eager attention to capture the attention weights
    model = AutoModel.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True,
        attn_implementation="eager"
    ).to(device).eval()

    processor = AutoImageProcessor.from_pretrained(model_name)

    num_registers = getattr(model.config, "num_register_tokens", 4)
    print(f"Model loaded. Number of register tokens identified: {num_registers}")

    transform = lambda img: processor(images=img, return_tensors="pt")['pixel_values'].squeeze()

    if hasattr(model, "vision_model"):
        num_layers = len(model.vision_model.encoder.layers)
    else:
        num_layers = len(model.layer)

    print(f"Success! Model loaded with {num_layers} layers.")

except Exception as e:
    print(f"Error loading model: {e}")
    print("Tip: If you run out of memory (OOM), try 'dinov3-vitl16' (Large, 24 layers) instead.")


### **2. Torchvision**
If you do not have access to `DinoV3` you can use free `DinoV2` from the `Torchvision` library.

In [ ]:
# TODO! Implement code for importing from TorchVision

## **Dataset Download**

In [ ]:
from sklearn.model_selection import train_test_split

### **1. Caltech**

In [ ]:
caltech_full = torchvision.datasets.Caltech101(
    root='./data',
    download=True
)

In [ ]:
total_images = len(caltech_full)
categories = caltech_full.categories
print(f"✅ Success! Loaded {total_images} high-resolution images.")
print(f"Total Categories: {len(categories)}\n")

In [ ]:
# torchvision's Caltech101 stores the numerical labels in the '.y' attribute
all_labels = caltech_full.y
all_indices = np.arange(total_images)

# Let's use 80% of the images to build our Bag of Words, and 20% to test it
train_idx, test_idx, train_labels, test_labels = train_test_split(
    all_indices,
    all_labels,
    test_size=0.20,
    random_state=42,
    stratify=all_labels
)

### **2. Your choice of the dataset**

In [ ]:
# TODO! Fill in with dataset to download

## **Feature Extraction**

In [ ]:
import os
from tqdm import tqdm

### **1. Method defition**

In [ ]:
def extract_and_save_features(indices, labels, dataset, split_name):
    features_list = []
    labels_list = []

    with torch.no_grad():
        for idx, label in tqdm(zip(indices, labels), total=len(indices), desc=f"Extracting {split_name.upper()}"):

            # Grab the image and FORCE it to be RGB (prevents grayscale crashes)
            img, _ = dataset[idx]
            img = img.convert("RGB")

            # Preprocess the image and send it to the GPU
            inputs = processor(images=img, return_tensors="pt").to(device)

            # Pass through the Flagship Model
            outputs = model(**inputs)

            # Sequence = [1 CLS Token] + [N Register Tokens] + [Image Patches]
            hidden_states = outputs.last_hidden_state
            patch_features = hidden_states[0, 1 + num_registers : ]

            # Move to CPU and cast to float16
            features_list.append(patch_features.cpu().to(torch.float16))
            labels_list.append(label)

    # Save the data to your hard drive using PyTorch's native save function
    filename = f"caltech101_{split_name}_flagship_features.pt"
    save_dict = {
        "features": features_list,
        "labels": labels_list
    }

    torch.save(save_dict, filename)

### **2. Extraction**

In [13]:
extract_and_save_features(train_idx, train_labels, caltech_full, "train")

Extracting TRAIN_SMALL: 100%|██████████| 6941/6941 [26:08<00:00,  4.42it/s]


In [14]:
extract_and_save_features(test_idx, test_labels, caltech_full, "test")

Extracting TEST_SMALL: 100%|██████████| 1736/1736 [06:20<00:00,  4.56it/s]


### **3. Loading the data**

In [15]:
train_data = torch.load("caltech101_train_flagship_features.pt", weights_only=False)
test_data = torch.load("caltech101_test_flagship_features.pt", weights_only=False)

In [16]:
caltech_train_features = [tensor.to(torch.float32).numpy() for tensor in train_data["features"]]
caltech_train_labels = train_data["labels"]

In [17]:
caltech_test_features = [tensor.to(torch.float32).numpy() for tensor in test_data["features"]]
caltech_test_labels = test_data["labels"]

### **4. Load them from Google Drive**

In [37]:
from google.colab import drive
import gc

In [38]:
print("🔌 Connecting to Google Drive...")
drive.mount('/content/drive')

🔌 Connecting to Google Drive...
Mounted at /content/drive


In [ ]:
BASE_PATH = '/content/drive/MyDrive/BC/features/'
TRAIN_FEATURES_FILE = BASE_PATH + 'caltech101_train_flagship_features.pt'
TEST_FEATURES_FILE = BASE_PATH + 'caltech101_test_flagship_features.pt'

In [ ]:
try:
    # ==========================================
    # 1. LOAD TRAIN FLAGSHIP DATA
    # ==========================================
    print(f"\n⏳ Loading Train Data from {TRAIN_FEATURES_FILE}...")
    raw_train_data = torch.load(TRAIN_FEATURES_FILE, map_location="cpu")

    # Check if it's a dictionary (containing both features and labels)
    if isinstance(raw_train_data, dict):
        caltech_train_features = raw_train_data.get('features', raw_train_data)
        caltech_train_labels = raw_train_data.get('labels', None)
    else:
        caltech_train_features = raw_train_data
        caltech_train_labels = None

    print(f"✅ Train Features loaded! (Type: {type(caltech_train_features)})")
    if caltech_train_labels is not None:
         print(f"✅ Train Labels also found inside the file!")

    del raw_train_data
    gc.collect() # Empty the RAM trash

    # ==========================================
    # 2. LOAD TEST FLAGSHIP DATA
    # ==========================================
    print(f"\n⏳ Loading Test Data from {TEST_FEATURES_FILE}...")
    raw_test_data = torch.load(TEST_FEATURES_FILE, map_location="cpu")

    if isinstance(raw_test_data, dict):
        caltech_test_features = raw_test_data.get('features', raw_test_data)
        caltech_test_labels = raw_test_data.get('labels', None)
    else:
        caltech_test_features = raw_test_data
        caltech_test_labels = None

    print(f"✅ Test Features loaded! (Type: {type(caltech_test_features)})")
    if caltech_test_labels is not None:
         print(f"✅ Test Labels also found inside the file!")

    del raw_test_data
    gc.collect()

    print("\n🎉 SUCCESS! Both Flagship files are fully loaded into memory.")

except RuntimeError as e:
    print(f"\n❌ CRASH: PyTorch ran out of RAM! Error: {e}")
    print("The Flagship file is too big for this Colab instance. Time to use the local slicer script!")
except FileNotFoundError as e:
    print(f"\n❌ ERROR: Colab couldn't find the file! Double check the name in Google Drive. Error: {e}")

## **Clustering**
We will be using [FAISS](https://ai.meta.com/tools/faiss/) from META because ....

In [18]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 25.2 MB/s eta 0:00:00


In [19]:
import faiss
import numpy as np
import time
import torch

### **Function definition**

In [22]:
def to_numpy_array(data):
    if torch.is_tensor(data):
        return data.cpu().numpy()
    elif isinstance(data, list):
        if torch.is_tensor(data[0]):
            return torch.stack(data).cpu().numpy()
        return np.array(data)
    return data

In [23]:
def flatten_labels(labels):
    if torch.is_tensor(labels):
        return labels.cpu().numpy().flatten()
    elif isinstance(labels, list):
        if torch.is_tensor(labels[0]):
            return torch.tensor(labels).cpu().numpy().flatten()
        return np.array(labels).flatten()
    return labels


### **Data Prep**

In [27]:
train_feats_np = to_numpy_array(caltech_train_features)
test_feats_np = to_numpy_array(caltech_test_features)

In [28]:
flat_train = train_feats_np.reshape(-1, train_feats_np.shape[-1]).astype(np.float32)

### **K-Means**

In [29]:
k = 1024
d = flat_train.shape[1]

In [30]:
start_time = time.time()
use_gpu = faiss.get_num_gpus() > 0
kmeans = faiss.Kmeans(d, k, niter=20, verbose=False, gpu=use_gpu)
kmeans.train(flat_train)

np.float64(4266483.5)

In [31]:
index = faiss.IndexFlatL2(d)
index.add(kmeans.centroids)

## **Histogram Generation**

### **Function Definition**

In [32]:
def build_histograms(features_array):
    hists = []
    for feats in features_array:
        feats_f32 = feats.reshape(-1, d).astype(np.float32)
        _, I = index.search(feats_f32, 1)
        hist = np.bincount(I.flatten(), minlength=k)
        hists.append(hist)
    return np.array(hists)

### **Creating them**

In [33]:
train_hists = build_histograms(train_feats_np)
train_l1 = train_hists / (np.sum(train_hists, axis=1, keepdims=True) + 1e-10)

In [34]:
test_hists = build_histograms(test_feats_np)
test_l1 = test_hists / (np.sum(test_hists, axis=1, keepdims=True) + 1e-10)

### **Converting Class Indexes to String**

In [47]:
categories = caltech_full.categories
print(f"First 5 categories available: {categories[:5]}")

integer_labels = flatten_labels(caltech_train_labels)

integer_labels = [int(lbl) for lbl in integer_labels]

string_labels = np.array([categories[i] for i in integer_labels])

print(f"Example of converted strings: {string_labels[:5]}")

np.savez_compressed(
    '/content/drive/MyDrive/BC/features/knn_reference_small_database.npz',
    histograms=train_l1,
    labels=string_labels
)

First 5 categories available: ['Faces', 'Faces_easy', 'Leopards', 'Motorbikes', 'accordion']
Example of converted strings: ['minaret' 'cellphone' 'stop_sign' 'grand_piano' 'Faces']


# **Saving the Model**

In [41]:
import json

In [39]:
np.save('/content/drive/MyDrive/BC/features/faiss_dictionary_small_k1024.npy', kmeans.centroids)

In [44]:
np.savez_compressed(
    '/content/drive/MyDrive/BC/features/knn_reference_small_database.npz',
    histograms=train_l1,
    labels=flatten_labels(caltech_train_labels)
)

In [42]:
hyperparameters = {
    "K_NN": 7,
    "DISTANCE_THRESHOLD": 0.5,
    "CHAOS_THRESHOLD": 3,
    "OOD_LABEL": -1,
    "num_clusters": k,
    "feature_dim": d
}

with open('/content/drive/MyDrive/BC/features/model_small_config.json', 'w') as f:
    json.dump(hyperparameters, f)